# 02 - Certificate collection

**Long-running and resumable. Start it, then leave this tab open and work
elsewhere.** This is the only step whose duration is not under our control, and
how much certificate coverage is achieved shapes every downstream decision.

Safe to interrupt at any point: the ledger holds the state, so re-running
resumes rather than restarting. Interrupting mid-batch loses at most one batch.

**Expect most DGA domains to fail with NXDOMAIN.** They were generated but never
registered. That is the finding, not a fault - it is why the lexical branch
carries DGA detection while the certificate branch carries live malicious
infrastructure from URLhaus and OpenPhish.

The ledger lives on local disk, not Drive: Drive's FUSE layer does not
implement POSIX advisory locking correctly and SQLite depends on it, so a
long-running writer on a mounted `.db` can corrupt exactly when the session
drops. A consistent backup is copied to Drive every few thousand probes.

In [ ]:
# --- standard header ---
from google.colab import drive
drive.mount('/content/drive')

import os, sys, subprocess, getpass
REPO = '/content/secure-dns-trust-ai'
URL  = 'github.com/sandesh20lamichhane/secure-dns-trust-ai.git'
if os.path.isdir(REPO):
    subprocess.run(['git','-C',REPO,'pull','-q'], check=False)
else:
    TOKEN = getpass.getpass('GitHub PAT: ')
    subprocess.run(['git','clone','-q',f'https://{TOKEN}@{URL}',REPO], check=True)
sys.path.insert(0, REPO)
os.environ['DNSTRUST_CONFIG_DIR'] = f'{REPO}/configs'

from src.utils import config, manifest, seeds
P = config.paths(); config.ensure_tree(P); seeds.set_all(42)
print('repo', manifest.git_sha(REPO))

In [ ]:
!pip -q install dnspython cryptography pyarrow zstandard

## One-time cleanup

An earlier version of `ensure_tree` created the ledger path as a *directory*.
SQLite then fails with a misleading "unable to open database file". This
removes the stray directory if it exists; harmless once it is gone.

In [ ]:
import shutil, os
led = P['local']['ledger']
if os.path.isdir(led):
    shutil.rmtree(led); print('removed stray directory:', led)
else:
    print('ledger path is clean:', led)

## Open the ledger

Statuses are deliberately granular. `no_certificate` and `nxdomain` are
findings; `timeout` and `connection_error` are artefacts to retry. Separating
them is what allows "this host has no TLS" to become a feature rather than
being confused with "the probe failed".

In [ ]:
import pandas as pd
from src.collect.ledger import Ledger
from src.collect import tls_prober
from src.utils.logging_setup import get_logger

log = get_logger('collect_tls', P['artifacts']['logs'])
ledger = Ledger(P['local']['ledger'],
                drive_backup=f"{P['artifacts']['logs']}/certificate_ledger_backup.db")
restored = ledger.restore_from_backup()      # rebuild after a fresh runtime
print('restored from Drive backup:', restored)
print(ledger.summary())

In [ ]:
universe = pd.read_parquet(f"{P['data']['interim']}/probe_universe.parquet")
n = ledger.enqueue(universe[['domain','source','label']]
                   .assign(label=universe['label'].astype(str))
                   .to_dict('records'))
print('newly enqueued:', n)
print('pending now   :', len(ledger.pending()))
print(ledger.summary())

## Run the probe

`max_batches` bounds one cell execution so the notebook does not appear frozen
for hours. Re-run this cell as many times as needed - it picks up exactly where
it stopped.

50k domains at 2,000 per batch is 25 batches. Each batch takes roughly five to
ten minutes depending on how many hosts time out, so plan on several runs of
this cell.

In [ ]:
summary = tls_prober.run_collection(
    ledger,
    out_dir=f"{P['data']['collected']}/tls_probe",
    batch_size=2000,
    concurrency=100,
    max_batches=5,          # raise once you have seen it work
    logger=log)
print(summary)

In [ ]:
# Progress at a glance. Re-run the probe cell until _complete_pct reaches 100.
sm = ledger.summary()
for k, v in sorted(sm.items()):
    print(f'{k:20s} {v}')

## When collection is complete

`retire_exhausted` marks transient failures past the retry cap as `abandoned`
so they stop being re-probed. Run it only once the probe cell reports no
remaining pending work.

In [ ]:
ledger.retire_exhausted()
print(ledger.summary())
ledger.backup()
print('ledger backed up to Drive')

## Coverage report

The number that matters for the paper: what fraction of each class actually
yielded a certificate. If the malicious side is dominated by `nxdomain`, that
is the empirical justification for the fusion design and belongs in the
results, not in a footnote.

In [ ]:
from src.utils.io import read_shards
import pandas as pd

tls = read_shards(f"{P['data']['collected']}/tls_probe", 'tls_probe')
print('certificates collected:', len(tls))

if len(tls):
    got = set(tls['domain'])
    universe['has_cert'] = universe['domain'].isin(got)
    cov = (universe.groupby(['label','source'])['has_cert']
                   .agg(n='count', with_cert='sum'))
    cov['rate'] = (cov['with_cert']/cov['n']).round(4)
    display(cov)

    q = pd.read_sql('SELECT status, label, COUNT(*) n FROM domains GROUP BY status, label',
                    ledger.conn)
    display(q.pivot(index='status', columns='label', values='n').fillna(0).astype(int))

In [ ]:
# Sanity check on what was actually captured
if len(tls):
    display(tls[['domain','issuer_org','is_free_ca','validity_days','cert_age_days',
                 'is_self_signed','is_expired','san_count','key_algorithm']].head(10))
    print('\nissuers:'); print(tls['issuer_org'].value_counts().head(10))

## Certificate Transparency (optional, run separately)

crt.sh is slow and rate-sensitive. Run it in a *second* Colab tab rather than
competing with the TLS probe for bandwidth. It adds issuance history -
notably first-seen date and issuance count, which are strong signals for
freshly registered malicious domains.

Only worth running for domains that actually resolved; CT lookups on
never-registered DGA domains return nothing.

In [ ]:
from src.collect import crtsh_client
from src.utils.io import ShardWriter
import time

live = list(tls['domain']) if len(tls) else []
print('domains with certificates to look up in CT:', len(live))

with ShardWriter(f"{P['data']['collected']}/crtsh", 'crtsh') as w:
    for i, d in enumerate(live[:2000]):
        status, rows = crtsh_client.fetch(d)
        if status == 'success':
            w.add(crtsh_client.summarise(d, rows))
        time.sleep(0.5)
        if i and i % 100 == 0:
            print(i, 'done')
print('CT collection finished')

---

**While this runs:** `04_split_creation`, then `03_feature_engineering`.
Feature engineering must come after split creation - the n-gram model is fitted
on training benign domains only, and fitting it on the full corpus would leak
the test distribution into the features.